Đọc dữ liệu

In [1]:
# Nạp thư viện os (Operating System) — dùng để tương tác với hệ điều hành (ví dụ: quản lý tệp tin, nối đường dẫn thư mục, kiểm tra tệp tồn tại).
import os
# Nạp thư viện numpy và đặt tên viết tắt là np — dùng để xử lý tính toán đại số tuyến tính, mảng nhiều chiều (array) và các phép toán số học tốc độ cao.
import numpy as np
# Nạp thư viện pandas và đặt tên viết tắt là pd — dùng để làm việc với dữ liệu dạng bảng (DataFrame), hỗ trợ đọc/ghi tệp CSV, Excel và tiền xử lý dữ liệu
import pandas as pd

# Ghép đường dẫn giữa thư mục 'path' (từ kagglehub) và tên tệp CSV
file_path = os.path.join(
    "..", "..", "data", "raw", "Diamonds Prices2022.csv"
)

# Đọc tệp CSV vào biến 'diamonds' (DataFrame) với dấu phân cách các cột là dấu phẩy (,)
diamonds = pd.read_csv(file_path, sep=',', index_col=0)

# In kích thước của tập dữ liệu (số dòng, số cột)
print(f"Kích thước tập dữ liệu ban đầu: {diamonds.shape}\n")

# Hiển thị thông tin tổng quan
diamonds.info()

# In tiêu đề thông báo cho phần hiển thị 5 dòng đầu
print("\n--- 5 dòng đầu tiên ---")
# Hiển thị 5 dòng dữ liệu đầu tiên
print(diamonds.head())

# In tiêu đề thông báo cho phần kiểm tra các giá trị phân loại
print("\n--- Giá trị phân loại của các thuộc tính dạng chuỗi ---")
# Lấy và in danh sách các giá trị duy nhất (không trùng lặp) của cột 'cut' (vết cắt)
print(f"Cut: {diamonds['cut'].unique()}")
# Lấy và in danh sách các giá trị duy nhất của cột 'color' (màu sắc)
print(f"Color: {diamonds['color'].unique()}")
# Lấy và in danh sách các giá trị duy nhất của cột 'clarity' (độ tinh khiết)
print(f"Clarity: {diamonds['clarity'].unique()}")

# In tiêu đề thông báo cho phần thống kê mô tả
print("\n--- Thống kê mô tả ---")
# Tính toán các chỉ số thống kê cơ bản cho các cột số
print(diamonds.describe())

Kích thước tập dữ liệu ban đầu: (53943, 10)

<class 'pandas.DataFrame'>
RangeIndex: 53943 entries, 1 to 53943
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   carat    53943 non-null  float64
 1   cut      53943 non-null  str    
 2   color    53943 non-null  str    
 3   clarity  53943 non-null  str    
 4   depth    53943 non-null  float64
 5   table    53943 non-null  float64
 6   price    53943 non-null  int64  
 7   x        53943 non-null  float64
 8   y        53943 non-null  float64
 9   z        53943 non-null  float64
dtypes: float64(6), int64(1), str(3)
memory usage: 4.1 MB

--- 5 dòng đầu tiên ---
   carat      cut color clarity  depth  table  price     x     y     z
1   0.23    Ideal     E     SI2   61.5   55.0    326  3.95  3.98  2.43
2   0.21  Premium     E     SI1   59.8   61.0    326  3.89  3.84  2.31
3   0.23     Good     E     VS1   56.9   65.0    327  4.05  4.07  2.31
4   0.29  Premium     I     VS2   

Tiền xử lý dữ liệu

In [2]:
# Nạp công cụ OrdinalEncoder từ thư viện scikit-learn để mã hóa các biến phân loại có thứ tự
from sklearn.preprocessing import OrdinalEncoder

# 1: Loại bỏ thuộc tính không mang thông tin
# Kiểm tra xem cột số thứ tự thừa 'Unnamed: 0' có tồn tại trong DataFrame hay không
if 'Unnamed: 0' in diamonds.columns:
    # Nếu có, xóa bỏ cột này khỏi tập dữ liệu
    diamonds = diamonds.drop(columns=['Unnamed: 0'])

# 2: Loại bỏ dữ liệu nhiễu (kích thước x, y, z <= 0)
# Lấy tổng số dòng dữ liệu ban đầu trước khi lọc
initial_rows = len(diamonds)

# Lọc giữ lại các dòng có cả 3 kích thước (x, y, z) đều lớn hơn 0 và tạo bản sao dữ liệu mới
diamonds = diamonds[(diamonds['x'] > 0) & (diamonds['y'] > 0) & (diamonds['z'] > 0)].copy()

# Tính số lượng dòng dữ liệu bị loại bỏ (bằng số dòng ban đầu trừ số dòng sau khi lọc)
invalid_dimensions = initial_rows - len(diamonds)

# In số lượng dòng nhiễu đã bị loại bỏ ra màn hình (thêm \n ở đầu để cách dòng trên)
print(f"\nĐã loại bỏ {invalid_dimensions} dòng dữ liệu nhiễu (x, y, z <= 0).")

# 3: Loại bỏ các dòng trùng lặp hoàn toàn
# Đếm tổng số dòng dữ liệu bị lặp lại hoàn toàn trong tập dữ liệu
num_duplicates = diamonds.duplicated().sum()

# Xóa bỏ các dòng trùng lặp, chỉ giữ lại 1 bản ghi đại diện
diamonds = diamonds.drop_duplicates()

# In số lượng dòng trùng lặp đã xóa ra màn hình (thêm \n ở đầu để cách với dòng trên)
print(f"\nĐã loại bỏ {num_duplicates} dòng trùng lặp.")

# 4: Mã hóa thứ tự (Ordinal Encoding) cho các thuộc tính phân loại
# Định nghĩa danh sách các mức độ chất lượng của vết cắt 'cut' theo thứ tự tăng dần (Fair là thấp nhất, Ideal là cao nhất)
cut_categories = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']

# Định nghĩa danh sách màu sắc 'color' theo thứ tự tăng dần từ kém nhất đến tốt nhất (J -> D)
color_categories = ['J', 'I', 'H', 'G', 'F', 'E', 'D']

# Định nghĩa danh sách độ tinh khiết 'clarity' theo thứ tự tăng dần từ kém nhất đến tốt nhất (I1 -> IF)
clarity_categories = ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']

# Khởi tạo đối tượng OrdinalEncoder với danh sách thứ tự đã thiết lập cho từng cột
encoder = OrdinalEncoder(categories=[cut_categories, color_categories, clarity_categories])

# chuyển đổi các cột dữ liệu dạng chữ (phân loại) thành dạng số dựa trên thứ tự đã định nghĩa
diamonds[['cut', 'color', 'clarity']] = encoder.fit_transform(diamonds[['cut', 'color', 'clarity']])

# 5: Kiểm tra dữ liệu sau tiền xử lý
# In kích thước mới của tập dữ liệu (số dòng, số cột) sau khi đã làm sạch và chuyển đổi
print(f"\nKích thước tập dữ liệu sau tiền xử lý: {diamonds.shape}\n")

# In tiêu đề thông báo hiển thị 5 dòng đầu
print("--- 5 dòng dữ liệu đầu sau tiền xử lý ---")

# Hiển thị 5 dòng dữ liệu đầu tiên sau tiền xử lý
print(diamonds.head())


Đã loại bỏ 20 dòng dữ liệu nhiễu (x, y, z <= 0).

Đã loại bỏ 148 dòng trùng lặp.

Kích thước tập dữ liệu sau tiền xử lý: (53775, 10)

--- 5 dòng dữ liệu đầu sau tiền xử lý ---
   carat  cut  color  clarity  depth  table  price     x     y     z
1   0.23  4.0    5.0      1.0   61.5   55.0    326  3.95  3.98  2.43
2   0.21  3.0    5.0      2.0   59.8   61.0    326  3.89  3.84  2.31
3   0.23  1.0    5.0      4.0   56.9   65.0    327  4.05  4.07  2.31
4   0.29  3.0    1.0      3.0   62.4   58.0    334  4.20  4.23  2.63
5   0.31  1.0    0.0      1.0   63.3   58.0    335  4.34  4.35  2.75


Chia dữ liệu

In [3]:
from sklearn.model_selection import KFold

# X là dữ liệu các thuộc tính
X = diamonds.drop(columns=["price"])
# y là dữ liệu nhãn
y = diamonds["price"]

# Chia dữ liệu thành 5 fold
kf = KFold(n_splits = 5, shuffle = True, random_state=42)

Training và kết quả đánh giá

In [ ]:
# Import mô hình hồi quy K-Nearest Neighbors (KNN) từ thư viện scikit-learn
from sklearn.neighbors import KNeighborsRegressor
# Import công cụ chuẩn hóa MinMaxScaler để đưa các đặc trưng về khoảng [0, 1]
from sklearn.preprocessing import MinMaxScaler
# Import các hàm tính toán độ đo đánh giá hiệu năng mô hình (MAE, MSE, R2)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# Import thư viện numpy để thực hiện các phép toán số học (tính trung bình, căn bậc hai)
import numpy as np
# Import thư viện pandas để hiển thị bảng kết quả từng fold gọn đẹp
import pandas as pd
# Import thư viện time để đo thời gian thực thi của mô hình
import time

# 1. THAM SỐ MỞ RỘNG (KHOẢNG CÁCH EUCLIDE)
# Tạo danh sách các giá trị số lượng láng giềng K cần thử nghiệm (từ 1 đến 200)
k_values = range(1, 201)
# Tạo danh sách các phương pháp tính trọng số láng giềng ('uniform': đều nhau, 'distance': theo khoảng cách)
weights_list = ['uniform', 'distance']

# Khởi tạo biến theo dõi sai số RMSE trung bình 5-Fold thấp nhất trên tập Test (gán ban đầu bằng dương vô cực)
best_rmse = float('inf')
# Khởi tạo biến lưu trữ từ điển bộ tham số tối ưu nhất
best_params = None

# Khởi tạo danh sách lưu trữ dữ liệu đã chia và chuẩn hóa sẵn của 5 fold để tránh lặp lại việc chuẩn hóa nhiều lần
folds_data = []
# Vòng lặp chia 5 fold dữ liệu từ đối tượng kf đã khởi tạo ở cell trước
for train_index, test_index in kf.split(X):
    # Tách tập đặc trưng huấn luyện X_train theo chỉ số train_index của fold hiện tại
    X_train = X.iloc[train_index]
    # Tách tập nhãn huấn luyện y_train theo chỉ số train_index của fold hiện tại
    y_train = y.iloc[train_index]
    # Tách tập đặc trưng kiểm tra X_test theo chỉ số test_index của fold hiện tại
    X_test = X.iloc[test_index]
    # Tách tập nhãn kiểm tra y_test theo chỉ số test_index của fold hiện tại
    y_test = y.iloc[test_index]

    # Khởi tạo đối tượng MinMaxScaler để chuẩn hóa dữ liệu riêng cho từng fold
    scaler = MinMaxScaler()
    # Học tham số Min-Max trên tập Train và chuyển đổi X_train sang khoảng [0, 1]
    X_train_scaled = scaler.fit_transform(X_train)
    # Chuyển đổi X_test dựa trên các tham số Min-Max đã học từ X_train (tránh rò rỉ dữ liệu)
    X_test_scaled = scaler.transform(X_test)

    # Lưu bộ dữ liệu đã chuẩn hóa của fold hiện tại vào danh sách folds_data
    folds_data.append((X_train_scaled, y_train, X_test_scaled, y_test))

# In thông báo bắt đầu quá trình duyệt tìm tham số tối ưu (K từ 1 đến 200)
print("--- Đang tìm kiếm tham số tối ưu cho KNN (K từ 1 đến 200, khoảng cách Euclide) ---")

# 2. DUYỆT QUA CÁC TỔ HỢP THAM SỐ TRÊN 5 FOLD
# Vòng lặp ngoài duyệt qua từng phương pháp tính trọng số trong danh sách weights_list
for w in weights_list:
    # Vòng lặp trong duyệt qua từng giá trị số lượng láng giềng K từ 1 đến 200
    for k in k_values:
        # Khởi tạo danh sách lưu trữ chỉ số RMSE trên tập Test qua 5 fold cho tổ hợp tham số hiện tại
        rmse_test_list = []

        # Vòng lặp duyệt qua từng fold dữ liệu đã được chuẩn hóa trong folds_data
        for X_train_scaled, y_train, X_test_scaled, y_test in folds_data:
            # Khởi tạo mô hình hồi quy KNN với số láng giềng k, trọng số w và khoảng cách Euclide (p=2)
            knn = KNeighborsRegressor(n_neighbors=k, weights=w, metric='minkowski', p=2, n_jobs=-1)
            # Huấn luyện mô hình KNN trên tập dữ liệu Train đã chuẩn hóa của fold hiện tại
            knn.fit(X_train_scaled, y_train)
            # Dự đoán giá trị đầu ra trên tập dữ liệu Test đã chuẩn hóa của fold hiện tại
            y_pred_test = knn.predict(X_test_scaled)
            # Tính chỉ số RMSE của fold hiện tại bằng căn bậc hai của MSE và thêm vào danh sách
            rmse_test_list.append(np.sqrt(mean_squared_error(y_test, y_pred_test)))

        # Tính giá trị trung bình của chỉ số RMSE qua 5 fold
        mean_rmse_test = np.mean(rmse_test_list)

        # Kiểm tra nếu RMSE trung bình của tổ hợp hiện tại nhỏ hơn mức RMSE thấp nhất từng ghi nhận
        if mean_rmse_test < best_rmse:
            # Cập nhật lại giá trị RMSE trung bình thấp nhất mới
            best_rmse = mean_rmse_test
            # Lưu lại từ điển chứa bộ tham số tối ưu tương ứng với mức RMSE thấp nhất đó
            best_params = {'n_neighbors': k, 'weights': w, 'metric': 'minkowski', 'p': 2}

# 3. ĐÁNH GIÁ CHI TIẾT TỪNG FOLD CỦA BỘ THAM SỐ TỐI ƯU NHẤT
# In ra màn hình bộ tham số tối ưu nhất tìm được trong khoảng K từ 1 đến 200
print("\nBỘ THAM SỐ TỐI ƯU NHẤT (K từ 1 đến 200):", best_params)

# Khởi tạo danh sách lưu trữ từ điển kết quả chi tiết của từng fold
fold_results = []

# Duyệt qua từng fold kèm theo chỉ số thứ tự fold_idx bắt đầu từ 1 đến 5
for fold_idx, (X_train_scaled, y_train, X_test_scaled, y_test) in enumerate(folds_data, start=1):
    # Khởi tạo mô hình KNN mới bằng cách giải nén các tham số từ từ điển best_params
    best_model = KNeighborsRegressor(**best_params)

    # Bắt đầu bấm giờ bằng hàm perf_counter() trước khi thực hiện huấn luyện và dự đoán
    start_time = time.perf_counter()
    # Huấn luyện mô hình tối ưu trên tập dữ liệu Train đã chuẩn hóa của fold hiện tại
    best_model.fit(X_train_scaled, y_train)
    # Dự đoán giá trị nhãn trên tập dữ liệu Test đã chuẩn hóa của fold hiện tại
    y_pred_test = best_model.predict(X_test_scaled)
    # Kết thúc bấm giờ và tính tổng thời gian chạy (fit + predict Test) cho fold hiện tại
    elapsed_time = time.perf_counter() - start_time

    # Dự đoán giá trị nhãn trên chính tập Train đã chuẩn hóa để đánh giá mức độ khớp dữ liệu
    y_pred_train = best_model.predict(X_train_scaled)

    # Thêm một từ điển chứa toàn bộ các chỉ số đánh giá của fold hiện tại vào danh sách fold_results
    fold_results.append({
        # Ghi tên thứ tự của fold (ví dụ: Fold 1, Fold 2,...)
        'Fold': f'Fold {fold_idx}',
        # Tính chỉ số sai số tuyệt đối trung bình (MAE) trên tập Train
        'MAE_Train': mean_absolute_error(y_train, y_pred_train),
        # Tính chỉ số sai số bình phương trung bình (MSE) trên tập Train
        'MSE_Train': mean_squared_error(y_train, y_pred_train),
        # Tính chỉ số căn bậc hai của MSE (RMSE) trên tập Train
        'RMSE_Train': np.sqrt(mean_squared_error(y_train, y_pred_train)),
        # Tính hệ số xác định R-squared (R^2) trên tập Train
        'R2_Train': r2_score(y_train, y_pred_train),
        # Tính chỉ số sai số tuyệt đối trung bình (MAE) trên tập Test
        'MAE_Test': mean_absolute_error(y_test, y_pred_test),
        # Tính chỉ số sai số bình phương trung bình (MSE) trên tập Test
        'MSE_Test': mean_squared_error(y_test, y_pred_test),
        # Tính chỉ số căn bậc hai của MSE (RMSE) trên tập Test
        'RMSE_Test': np.sqrt(mean_squared_error(y_test, y_pred_test)),
        # Tính hệ số xác định R-squared (R^2) trên tập Test
        'R2_Test': r2_score(y_test, y_pred_test),
        # Lưu thời gian thực thi (fit + predict Test) của fold hiện tại tính bằng giây
        'Time (s)': elapsed_time
    })

# Chuyển danh sách kết quả fold_results thành bảng DataFrame của thư viện pandas
df_folds = pd.DataFrame(fold_results)

# In tiêu đề thông báo bảng kết quả chi tiết từng fold (K từ 1 đến 200)
print("\n=== KẾT QUẢ CHI TIẾT TỪNG FOLD CỦA BỘ THAM SỐ TỐI ƯU (K: 1-200) ===")
# Làm tròn các số liệu trong bảng đến 4 chữ số thập phân và in toàn bộ bảng ra màn hình (ẩn cột chỉ số mặc định)
print(df_folds.round(4).to_string(index=False))

# In tiêu đề thông báo kết quả trung bình trên tập huấn luyện Train
print("\n=== KẾT QUẢ MÔ HÌNH TỐI ƯU TRÊN TẬP TRAIN (TRUNG BÌNH 5-FOLD, K: 1-200) ===")
# Tính giá trị trung bình cột MAE_Train qua 5 fold, làm tròn 4 chữ số thập phân và in ra
print("MAE :", round(df_folds['MAE_Train'].mean(), 4))
# Tính giá trị trung bình cột MSE_Train qua 5 fold, làm tròn 4 chữ số thập phân và in ra
print("MSE :", round(df_folds['MSE_Train'].mean(), 4))
# Tính giá trị trung bình cột RMSE_Train qua 5 fold, làm tròn 4 chữ số thập phân và in ra
print("RMSE:", round(df_folds['RMSE_Train'].mean(), 4))
# Tính giá trị trung bình cột R2_Train qua 5 fold, làm tròn 4 chữ số thập phân và in ra
print("R^2 :", round(df_folds['R2_Train'].mean(), 4))

# In tiêu đề thông báo kết quả trung bình trên tập kiểm tra Test
print("\n=== KẾT QUẢ MÔ HÌNH TỐI ƯU TRÊN TẬP TEST (TRUNG BÌNH 5-FOLD, K: 1-200) ===")
# Tính giá trị trung bình cột MAE_Test qua 5 fold, làm tròn 4 chữ số thập phân và in ra
print("MAE :", round(df_folds['MAE_Test'].mean(), 4))
# Tính giá trị trung bình cột MSE_Test qua 5 fold, làm tròn 4 chữ số thập phân và in ra
print("MSE :", round(df_folds['MSE_Test'].mean(), 4))
# Tính giá trị trung bình cột RMSE_Test qua 5 fold, làm tròn 4 chữ số thập phân và in ra
print("RMSE:", round(df_folds['RMSE_Test'].mean(), 4))
# Tính giá trị trung bình cột R2_Test qua 5 fold, làm tròn 4 chữ số thập phân và in ra
print("R^2 :", round(df_folds['R2_Test'].mean(), 4))
# Tính tổng thời gian thực thi của cả 5 fold, định dạng 4 chữ số thập phân và in ra
print(f"Tổng thời gian thực thi 5 fold (fit + predict Test): {df_folds['Time (s)'].sum():.4f} giây")
# Tính thời gian thực thi trung bình trên mỗi fold, định dạng 4 chữ số thập phân và in ra
print(f"Thời gian thực thi trung bình mỗi fold: {df_folds['Time (s)'].mean():.4f} giây")

--- Đang tìm kiếm tham số tối ưu cho KNN (K từ 101 đến 200, khoảng cách Euclide) ---

BỘ THAM SỐ TỐI ƯU NHẤT: {'n_neighbors': 101, 'weights': 'distance', 'metric': 'minkowski', 'p': 2}

=== KẾT QUẢ MÔ HÌNH TỐI ƯU TRÊN TẬP TRAIN (TRUNG BÌNH 5-FOLD) ===
MAE: 0.4263
MSE: 62.0639
RMSE: 7.8388
R^2: 1.0

=== KẾT QUẢ MÔ HÌNH TỐI ƯU TRÊN TẬP TEST (TRUNG BÌNH 5-FOLD) ===
MAE: 436.5083
MSE: 699389.942
RMSE: 836.007
R^2: 0.956
Tổng thời gian thực thi 5 fold (fit + predict Test): 4.7056 giây
